# 01 — Create a Bedrock Managed Knowledge Base with S3 Data Source

This notebook walks through the end-to-end flow of creating a **Bedrock Managed Knowledge Base (BMKB)** using Amazon Bedrock, with S3 as the data source.

**What "Bedrock Managed" means:** Bedrock handles the vector store for you — no need to provision OpenSearch, Pinecone, or any external vector DB. You just point it at your documents and query.

### What this notebook does

1. Configures your environment (region, bucket, models)
2. Creates the BMKB with all required IAM roles and policies
3. Ingests documents from S3
4. Queries the KB using Retrieve and AgenticRetrieveStream APIs
5. Cleans up all resources

### Prerequisites

- S3 bucket with documents to ingest (or the notebook will create one)
- AWS credentials with Bedrock and IAM permissions
- **Kernel:** Select `Python 3`

### Architecture

```
S3 Bucket (docs) ──► BMKB (Bedrock-managed vector store) ──► Retrieve / AgenticRetrieveStream
                         │
                         ├── IAM Role (auto-created)
                         ├── Embedding: Managed default (no extra cost)
                         └── Parsing: Smart Parsing
```

In [ ]:
%pip install --upgrade pip --quiet
%pip install -r ../requirements.txt --quiet

In [ ]:
# restart kernel
from IPython.core.display import HTML
HTML("<script>Jupyter.notebook.kernel.restart()</script>")

## Step 1 — Configuration

Update the values below to match your environment.

In [1]:
import boto3
import sys
import time
import os
import logging
import pprint

sys.path.insert(0, "..")

# Clients
s3_client = boto3.client('s3')
sts_client = boto3.client('sts')
session = boto3.session.Session()
region = session.region_name
account_id = sts_client.get_caller_identity()['Account']

logging.basicConfig(format='[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s', level=logging.INFO)
logger = logging.getLogger(__name__)

# Generate unique suffix for resource naming
suffix = time.strftime('%Y%m%d%H%M%S', time.localtime())[-7:]

# ── Configuration (update these) ─────────────────────────────────────
knowledge_base_name = f'bmkb-workshop-rag-{suffix}'
knowledge_base_description = 'BMKB workshop - S3 data source'
bucket_name = f'bedrock-bmkb-{suffix}-{account_id}'

# Embedding model — use None for the managed default (no extra cost)
# Or specify a custom Bedrock model (additional embedding cost applies)
embedding_model = None  # Managed default — optimized, no extra cost
# embedding_model = 'amazon.titan-embed-text-v2:0'  # Custom — additional cost
# Cross-region inference profile prefix depends on your region
region_prefix_map = {'us-': 'us', 'eu-': 'eu', 'ap-': 'apac'}
cris_prefix = next((v for k, v in region_prefix_map.items() if region.startswith(k)), 'us')
generation_model_arn = f'arn:aws:bedrock:{region}:{account_id}:inference-profile/{cris_prefix}.anthropic.claude-haiku-4-5-20251001-v1:0'
# Note: Newer models (Claude Haiku 4.5, Sonnet 4, etc.) require inference profile ARNs.
# Using a bare model ID will result in a ValidationException.

pp = pprint.PrettyPrinter(indent=2)

print(f'Region:     {region}')
print(f'Account:    {account_id}')
print(f'KB Name:    {knowledge_base_name}')
print(f'Bucket:     {bucket_name}')
print(f'Embedding:  {embedding_model or "managed default (no extra cost)"}')
print(f'Generation: {generation_model_arn}')

Region:     us-west-2
Account:    017444429555
KB Name:    bmkb-workshop-rag-9074712
Bucket:     bedrock-bmkb-9074712-017444429555
Embedding:  managed default (no extra cost)
Generation: arn:aws:bedrock:us-west-2:017444429555:inference-profile/us.anthropic.claude-haiku-4-5-20251001-v1:0


## Step 2 — Create S3 bucket and upload synthetic dataset

Check if the S3 bucket exists, create it if needed, then upload the Octank Financial 10K synthetic dataset.

In [2]:
# Check if bucket exists, create if not
try:
    s3_client.head_bucket(Bucket=bucket_name)
    print(f'Bucket {bucket_name} already exists')
except Exception:
    print(f'Creating bucket {bucket_name}')
    if region == 'us-east-1':
        s3_client.create_bucket(Bucket=bucket_name)
    else:
        s3_client.create_bucket(
            Bucket=bucket_name,
            CreateBucketConfiguration={'LocationConstraint': region}
        )

Creating bucket bedrock-bmkb-9074712-017444429555


In [3]:
# Upload Octank Financial 10K report to S3
file_to_upload = '../synthetic_dataset/octank_financial_10K.pdf'
print(f'Uploading {file_to_upload} to s3://{bucket_name}/')
s3_client.upload_file(file_to_upload, bucket_name, 'octank_financial_10K.pdf')
print('Done.')

Uploading ../synthetic_dataset/octank_financial_10K.pdf to s3://bedrock-bmkb-9074712-017444429555/
Done.


## Step 3 — Create the Bedrock Managed Knowledge Base

The `ManagedKnowledgeBase` utility handles everything:
- Creates/verifies the S3 bucket
- Creates an IAM execution role with scoped policies
- Creates a `MANAGED` type KB (Bedrock manages the vector store)
- Creates an S3 data source using the managed connector
- Waits for all resources to become active

In [4]:
from utils.managed_knowledge_base import ManagedKnowledgeBase

kb = ManagedKnowledgeBase(
    kb_name=knowledge_base_name,
    bucket_name=bucket_name,
    embedding_model=embedding_model,
    enable_logging=True,
    region_name=region,
    suffix=suffix,
)

print(f"\nKB ID: {kb.kb_id}")
print(f"DS ID: {kb.ds_id}")

kb_id = kb.kb_id
%store kb_id

[2026-06-19 07:47:23,783] p85694 {credentials.py:1392} INFO - Found credentials in shared credentials file: ~/.aws/credentials


Step 1 — Ensuring S3 buckets exist
  Bucket 'bedrock-bmkb-9074712-017444429555' already exists
Step 2 — Creating IAM role and policies
  Created role: AmazonBedrockExecutionRoleForKnowledgeBase_9074712
  Created policy: AmazonBedrockCloudWatchPolicyForKnowledgeBase_9074712
  Created policy: AmazonBedrockS3PolicyForKnowledgeBase_9074712
  Waiting for IAM propagation...
..........
Step 3 — Creating Bedrock Managed Knowledge Base
  Created KB: V9DVBZSDES
  KB status: ACTIVE
Step 4 — Creating 1 data source(s)
  Created data source 'bmkb-workshop-rag-9074712-s3-source': LZTCNAAISK
  Data source status: AVAILABLE
Setup complete — KB ID: V9DVBZSDES, 1 data source(s)
Step 5 — Enabling CloudWatch log delivery
  Created log group: /aws/vendedlogs/bedrock/knowledge-base/APPLICATION_LOGS/V9DVBZSDES
  Created delivery source: bedrock-kb-V9DVBZSDES
  Created delivery destination: bedrock-kb-dest-V9DVBZSDES
  Log delivery enabled (ID: vbSWWyT3cDtGoZIA)

KB ID: V9DVBZSDES
DS ID: LZTCNAAISK
Stored 'kb_

## Step 4 — Ingest documents

Start an ingestion job to crawl S3 and index documents into the managed vector store.

In [5]:
job = kb.start_ingestion_job()

Ingestion job started for LZTCNAAISK: KZJP0VLBFL
  ... IN_PROGRESS — scanned=0, indexed=0, failed=0
  ... IN_PROGRESS — scanned=1, indexed=0, failed=0

Ingestion COMPLETE (LZTCNAAISK)
  Source files: 1
  Added:        1
  Modified:     0
  Deleted:      0
  Failed:       0


## Step 5 — Query the Knowledge Base

BMKB supports two retrieval APIs:

| API | `generate_response` | What you get |
|-----|--------------------|--------------|
| **Retrieve** | N/A | Raw chunks with scores — build your own prompt |
| **AgenticRetrieveStream** | `False` | Deduplicated chunks (agentic sub-query decomposition) |
| **AgenticRetrieveStream** | `True` (default) | Chunks + a synthesized answer with citations |

> **Note:** `RetrieveAndGenerate` is not supported for Managed Knowledge Bases. Use `AgenticRetrieveStream` with `generate_response=True` for the same functionality.

### 5a. Retrieve API
Returns raw source chunks with relevance scores. Use this when you want to build your own prompt.

In [ ]:
response = kb.retrieve("What is Octank Financial's total revenue?", num_results=5)

print("=== Retrieve API ===")
for i, res in enumerate(response.get("retrievalResults", []), 1):
    score = res["score"]
    text = res["content"]["text"][:120]
    uri = res.get("location", {}).get("s3Location", {}).get("uri", "N/A")
    print(f"{i}. score={score:.4f} | {text}...")
    print(f"   source: {uri}")
print(f"\nTotal: {len(response.get('retrievalResults', []))} chunks")

### 5b. AgenticRetrieveStream (with generation)
Decomposes query, retrieves chunks, and generates a synthesized answer.

In [ ]:
result = kb.agentic_retrieve_stream(
    query="What are the key risk factors mentioned in Octank Financial's 10K report?",
    model_arn=generation_model_arn,
    generate_response=True,
)

print('=== AgenticRetrieveStream (with generation) ===')
print(f"Answer:\n{result['generated_response']['answer']}")
print(f"\nCitations: {len(result['generated_response'].get('citations', []))}")
for i, citation in enumerate(result['generated_response'].get('citations', [])[:3], 1):
    refs = citation.get('references', [])
    start = citation.get('startIndex', 0)
    end = citation.get('endIndex', 0)
    print(f'  Citation {i} [chars {start}-{end}]: {len(refs)} reference(s)')
    for ref in refs[:3]:
        idx = ref.get('resultIndex', -1)
        if idx >= 0 and idx < len(result['results']):
            chunk_text = result['results'][idx]['content']['text'][:100]
            print(f'    - result[{idx}]: {chunk_text}...')


### 5c. AgenticRetrieveStream API
Decomposes complex queries into sub-queries, runs multiple retrievals, and returns deduplicated chunks via streaming. Works with a single KB.

In [ ]:
result = kb.agentic_retrieve_stream(
    query="What are the main business segments and how did each perform financially?",
    model_arn=generation_model_arn,
    max_results=10,
    max_iterations=3,
)

print("=== AgenticRetrieveStream API ===")
print(f"Trace events: {len(result['traces'])}")
for t in result['traces']:
    attrs = t.get('attributes', {})
    print(f"  [{attrs.get('step', '?')}] {attrs.get('status', '')}")

print(f"\nFinal: {len(result['results'])} deduplicated chunks")
for i, r in enumerate(result['results'][:5], 1):
    text = r['content']['text'][:120]
    print(f"  {i}. {text}...")
# Show generated response if available
if result.get('generated_response'):
    print(f"\nGenerated Answer:\n{result['generated_response']['answer']}")
    citations = result['generated_response'].get('citations', [])
    print(f"\nCitations: {len(citations)}")
    for i, citation in enumerate(citations[:3], 1):
        refs = citation.get('references', [])
        start = citation.get('startIndex', 0)
        end = citation.get('endIndex', 0)
        print(f'  Citation {i} [chars {start}-{end}]: {len(refs)} reference(s)')
        for ref in refs[:2]:
            idx = ref.get('resultIndex', -1)
            if idx >= 0 and idx < len(result['results']):
                print(f'    - result[{idx}]: {result["results"][idx]["content"]["text"][:80]}...')


### 5d. Try your own queries

In [ ]:
# Try your own query
QUERY = "What is Octank Financial's strategy for growth?"

result = kb.agentic_retrieve_stream(
    query=QUERY,
    model_arn=generation_model_arn,
    generate_response=True,
)
print(result['generated_response']['answer'])


## Step 6 — Add more data sources (optional)

You can add additional data sources to the same KB. For example, add a Web Crawler source alongside the S3 source:

In [ ]:
# Uncomment to add a Web Crawler data source to this KB
# web_ds_id = kb.add_data_source({
#     'type': 'WEB',
#     'seed_urls': ['https://docs.aws.amazon.com/bedrock/latest/userguide/knowledge-base.html'],
#     'crawl_depth': 2,
#     'sync_scope': 'SUB_DOMAINS',
# })
# print(f'Added Web Crawler data source: {web_ds_id}')
#
# # Ingest the new data source
# kb.start_ingestion_job(ds_id=web_ds_id)
#
# # View all data sources
# print(f'Data sources: {kb.data_sources}')

## Step 7 — Cleanup

Delete all resources created by this notebook. The utility handles the correct ordering:
data sources → KB → IAM roles/policies → (optionally) S3 bucket.

**Options:**
- `delete_iam=True` (default) — removes the IAM role and policies
- `delete_s3_bucket=True` — also deletes the S3 bucket and all objects

> Only run this when you're done experimenting.

In [ ]:
# Uncomment to delete everything
print('===============================Deleting Knowledge Base and associated resources==============================')
# kb.delete_kb(delete_iam=True, delete_s3_bucket=True)

In [ ]:
# # Alternative: delete by KB ID only (if you lost the object)
# import sys
# sys.path.insert(0, "..")

# from utils.managed_knowledge_base import ManagedKnowledgeBase
# ManagedKnowledgeBase.delete_kb_by_id("YOUR_KB_ID", region_name="us-west-2")

## Summary

| What | Details |
|---|---|
| KB Type | `MANAGED` — Bedrock handles the vector store |
| Data Source | S3 via `MANAGED_KNOWLEDGE_BASE_CONNECTOR` |
| Parsing | Smart Parsing (default, managed by Bedrock) |
| Embedding | Managed default (no extra cost) or custom Bedrock model |
| IAM | Auto-created scoped role + 3 policies |
| Retrieval | `Retrieve` (raw chunks), `AgenticRetrieveStream` (agentic retrieval with optional generation) |
